In [ ]:
def fetch_metadata_from_gb(input_file):
    '''
    For each entry in GenBank file retrieves the following data:

    # Identifiers
    Accession
    GenBank title (DEFINITION)

    # Classification
    Organism name
    Species
    Isolate
    Strain
    Family
    Lineage

    #Sequence quality
    Length
    Strand

    Num of annotated CDS

    ORF1a
    ORF1b
    ORF2
    coords
    
    #Sources
    Country
    Tissue/Specimen/Source
    Collection date
    Release date
    Submitters
    
    '''

    '''
     dictionary
     entries_data[GenBank Accession] = {}
     entries_data[GenBank Accession].keys() = [
                                                'Version',
                                                'GenBank title'
                                                'Organism name',
                                                'Species',
                                                'Family',
                                                'Virus Lineage',
                                                'Length',
                                                'Isolate',
                                                'Strain',
                                                'Strand',
                                                'Geo location'
                                                'Country',
                                                'Tissue/Specimen/Source',
                                                'Host',
                                                'Collection date',
                                                'Release date',
                                                'Submitters',
                                                'Sequencing Technology'
                                                'CDS number',
                                                'Artificial',
                                                'UNVERIFIED'
                                                ]
    '''
    entries_data = {}

    source_qualifiers = ["strain", "isolate", "geo_loc_name", "country", "collection_date", "isolation_source", "host"]
    source_qualifiers_columns = ["Strain", "Isolate", "Geo location", "Country", "Collection date", "Tissue/Specimen/Source", "Host"]

    entries = SeqIO.parse(os.path.abspath(input_file), "genbank")

    for entry in entries:
        entries_data[entry.name] = {}
        entries_data[entry.name]['Version'] = entry.id
        entries_data[entry.name]['GenBank title'] = entry.description

        entries_data[entry.name]['Release date'] = entry.annotations['date']
        entries_data[entry.name]['Organism name'] = entry.annotations['organism']
        
        entries_data[entry.name]['Virus Lineage'] = ';'.join(entry.annotations['taxonomy'])
        num_taxa = len(entry.annotations['taxonomy'])
        if num_taxa == 9:
            entries_data[entry.name]['Species'] = entry.annotations['taxonomy'][-1]
        else:
             entries_data[entry.name]['Species'] = 'NA'
        entries_data[entry.name]['Family'] = entry.annotations['taxonomy'][6]
        
        entries_data[entry.name]['Length'] = len(entry)

        for reference in entry.annotations['references']:
            if reference.title == 'Direct Submission':
                entries_data[entry.name]['Submitters'] = reference.authors

        count_cds = 0
        for feature in entry.features:
            if feature.type == 'source':
                for qualif, colname in zip(source_qualifiers, source_qualifiers_columns):
                    if qualif in feature.qualifiers:
                        entries_data[entry.name][colname] = feature.qualifiers[qualif][0]
                    else:
                        entries_data[entry.name][colname] = 'NA'
                if 'environmental_sample' in feature.qualifiers:
                    entries_data[entry.name]['Environmental'] = 'Yes'
                else:
                    entries_data[entry.name]['Environmental'] = 'No'
                    
            if feature.type == 'CDS':
                
                count_cds +=1
                
        entries_data[entry.name]['CDS count'] = count_cds


        artificial_condition1 = ("patent" in entry.description.lower()) or ("oligonucleotide" in entry.description.lower())
        artificial_condition2 = "crispr" in entry.description.lower()
        artificial_condition3 = "method" in entry.description.lower()
        artificial_condition4 = ("modified microbial nucleic acid" in entry.description.lower()) or ("nonfunctional" in entry.description.lower())

        
        
        if artificial_condition1 or artificial_condition2 or artificial_condition3 or artificial_condition4:
            entries_data[entry.name]['Artificial'] = "Yes"
        else:
            entries_data[entry.name]['Artificial'] = "No"


        if 'UNVERIFIED' in entry.description:
            entries_data[entry.name]['UNVERIFIED'] = "Yes"
        else:
            entries_data[entry.name]['UNVERIFIED'] = "No"

    meta_dataframe = pd.DataFrame.from_dict(entries_data, orient='index')
    
    meta_dataframe.to_csv(os.path.splitext(os.path.abspath(input_file))[0] + '.csv')
    #country_map - file with abbreviations of countries
    return meta_dataframe